# B4 — Généraliser et convaincre

| Projet | Étape | Durée | Modules réinvestis |
|---|---|---|---|
| B — Influence dans un réseau social | 4 sur 4 | 4 h | M1, M4, M7 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

Une association qui étudie les réseaux d'entraide d'un quartier a entendu parler de votre travail. Elle ne s'intéresse pas au club de karaté : elle veut savoir **comment repérer les personnes influentes dans n'importe quel réseau**, et disposer d'un outil pour le faire.

Pour en tirer des conclusions générales, il faut d'autres réseaux. Le projet en fournit deux, très différents du club :

- les **mariages entre grandes familles florentines** vers 1430, étudiés par le sociologue John Padgett pour comprendre l'ascension des Médicis ;
- les **personnages des *Misérables*** de Victor Hugo, liés quand ils apparaissent dans un même chapitre (données de Donald Knuth).

**Les questions de cette étape : quelles conclusions valent pour tous les réseaux, et lesquelles dépendent du réseau ? Comment livrer un outil fiable, et comment argumenter une affirmation comme « X est le plus influent » ?**

**Activité** : Avant de calculer, notez ce qui distingue ces trois réseaux : taille, nature des liens, poids, ce que voudrait dire « influent » dans chacun.

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

In [ ]:
pari_famille_la_plus_influente = "Strozzi"       # (modifiez)
pari_personnage_le_plus_influent = "Javert"      # (modifiez)

## ③ Abstraire : une mesure de plus, l'intermédiarité

Les mesures des étapes B1 et B2 récompensent les membres bien entourés. Une autre forme d'influence consiste à **contrôler les passages** : être sur le chemin entre des groupes qui ne communiquent pas autrement. L'**intermédiarité** d'un membre $k$ est la part des plus courts chemins entre deux autres membres qui passent par $k$ :

$$
g_k = \frac{1}{\binom{n-1}{2}} \sum_{\substack{i < j \\ i, j \neq k}} \frac{\sigma_{ik}\,\sigma_{kj}}{\sigma_{ij}} \quad \text{pour les paires où } d_{ik} + d_{kj} = d_{ij},
$$

où $d_{ij}$ est la distance et $\sigma_{ij}$ le nombre de plus courts chemins entre $i$ et $j$. Les puissances de $A$ donnent les deux : la distance est la première puissance $k$ pour laquelle $(A^k)_{ij} > 0$, et $\sigma_{ij}$ est alors égal à $(A^k)_{ij}$.

**Activité** : Dans un réseau en ligne $1 - 2 - 3$, quelle est l'intermédiarité de chaque membre ? Et dans une étoile à 5 branches, pour le centre et pour une extrémité ?

Indiquez vos réponses et justifiez-les.

## ④ Calculer

### L'outil : un module de mesures d'influence

Tout le travail des étapes B1 à B3 tient en quelques fonctions. Chacune vérifie ses entrées : une matrice qui ne décrit pas un réseau non orienté, un membre isolé ou un réseau en plusieurs morceaux sont signalés clairement.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from reseaux import charger, dessiner

def verifier_adjacence(A):
    """Vérifie qu'une matrice décrit un réseau non orienté : carrée, symétrique, positive, sans boucle."""
    A = np.asarray(A, dtype=float)
    if A.ndim != 2 or A.shape[0] != A.shape[1]:
        raise ValueError("La matrice d'adjacence doit être carrée.")
    if not np.allclose(A, A.T):
        raise ValueError("La matrice d'adjacence doit être symétrique (liens non orientés).")
    if (A < 0).any():
        raise ValueError("Les poids des liens doivent être positifs.")
    if np.any(np.diag(A) != 0):
        raise ValueError("Un membre ne peut pas être lié à lui-même.")
    return A


def distances_et_chemins(A):
    """Distances (longueur du plus court chemin) et nombre de plus courts chemins, par puissances de A."""
    B = (verifier_adjacence(A) > 0).astype(float)
    n = len(B)
    D = np.full((n, n), np.inf)
    S = np.zeros((n, n))
    np.fill_diagonal(D, 0)
    np.fill_diagonal(S, 1)
    P = np.eye(n)
    for k in range(1, n):
        P = P @ B                                   # P[i, j] = nombre de chemins de longueur k
        nouveaux = (P > 0) & np.isinf(D)
        D[nouveaux] = k
        S[nouveaux] = P[nouveaux]                   # les premiers chemins trouvés sont les plus courts
        if not nouveaux.any():
            break
    return D, S


def intermediarite(A):
    """Part des plus courts chemins entre deux autres membres qui passent par chaque membre."""
    D, S = distances_et_chemins(A)
    n = len(D)
    resultat = np.zeros(n)
    for k in range(n):
        for i in range(n):
            for j in range(i + 1, n):
                if k in (i, j) or not np.isfinite(D[i, j]):
                    continue
                if D[i, k] + D[k, j] == D[i, j]:
                    resultat[k] += S[i, k] * S[k, j] / S[i, j]
    paires = (n - 1) * (n - 2) / 2
    return resultat / paires


def centralites(A, alpha_relatif=0.5, d=0.85):
    """Tableau des mesures d'influence de chaque membre (une ligne par membre)."""
    A = verifier_adjacence(A)
    n = len(A)
    if n < 3:
        raise ValueError("Il faut au moins trois membres.")
    degre = A.sum(axis=1)
    if (degre == 0).any():
        raise ValueError("Le réseau contient un membre isolé : les mesures récursives ne sont pas définies.")
    valeurs, vecteurs = np.linalg.eigh(A)
    lam1 = valeurs[-1]
    propre = np.abs(vecteurs[:, -1])
    katz = np.linalg.solve(np.eye(n) - alpha_relatif / lam1 * A, np.ones(n)) - 1
    M = A / degre
    pagerank = np.ones(n) / n
    for _ in range(500):
        pagerank = d * M @ pagerank + (1 - d) / n
    D, _ = distances_et_chemins(A)
    if np.isinf(D).any():
        raise ValueError("Le réseau n'est pas connexe : traitez chaque partie séparément.")
    return pd.DataFrame({
        "degré": degre,
        "vecteur propre": propre / propre.sum(),
        "Katz": katz / katz.sum(),
        "PageRank": pagerank,
        "proximité": (n - 1) / D.sum(axis=1),
        "intermédiarité": intermediarite(A),
    })


def coupe_spectrale(A):
    """Sépare le réseau en deux groupes selon le signe du vecteur de Fiedler ; renvoie (groupe, vecteur)."""
    A = verifier_adjacence(A)
    L = np.diag(A.sum(axis=1)) - A
    _, vecteurs = np.linalg.eigh(L)
    v2 = vecteurs[:, 1]
    if v2[0] < 0:                                   # convention : le premier membre est dans le groupe 0
        v2 = -v2
    return (v2 < 0).astype(int), v2


def portee(A, departs, p=0.2, etapes=3, essais=300, graine=0):
    """Nombre moyen de membres touchés par une rumeur partie de `departs` (modèle en cascade)."""
    A = verifier_adjacence(A)
    if not 0 <= p <= 1:
        raise ValueError("La probabilité de transmission doit être comprise entre 0 et 1.")
    voisins = [np.nonzero(A[i])[0] for i in range(len(A))]
    rng = np.random.default_rng(graine)
    total = 0
    for _ in range(essais):
        informes, nouveaux = set(departs), list(departs)
        for _ in range(etapes):
            suivants = []
            for i in nouveaux:
                for j in voisins[i]:
                    if j not in informes and rng.random() < p:
                        informes.add(j)
                        suivants.append(j)
            nouveaux = suivants
        total += len(informes)
    return total / essais

In [ ]:
# Tests rapides sur des réseaux dont on connaît la réponse
etoile = np.zeros((5, 5)); etoile[0, 1:] = etoile[1:, 0] = 1
assert (centralites(etoile).idxmax() == 0).all(), "le centre d'une étoile domine toutes les mesures"
chemin = np.array([[0, 1, 0], [1, 0, 1], [0, 1, 0]], dtype=float)
assert np.allclose(intermediarite(chemin), [0, 1, 0])
cycle = np.roll(np.eye(6), 1, axis=1); cycle = cycle + cycle.T
assert np.allclose(centralites(cycle).std(), 0), "dans un cycle, tous les membres sont équivalents"
try:
    centralites(np.array([[0, 1, 0], [0, 0, 1], [0, 1, 0]], dtype=float))
    raise AssertionError("une matrice non symétrique aurait dû être refusée")
except ValueError:
    pass
print("Tests réussis")

### Trois réseaux, six mesures

In [ ]:
resultats = {}
for nom in ["karate", "florence", "miserables"]:
    membres, _, A = charger(nom)
    c = centralites(A)
    c.index = membres["id"]
    resultats[nom] = (membres, A, c)
    print(f"\n{nom} : {len(A)} membres, {int(A.sum() / 2)} liens")
    podium = pd.DataFrame({m: c[m].sort_values(ascending=False).index[:3].astype(str) for m in c.columns},
                          index=["1er", "2e", "3e"])
    print(podium.to_string())

### Les mesures s'accordent-elles, réseau par réseau ?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, (nom, (_, _, c)) in zip(axes, resultats.items()):
    corr = c.rank().corr()
    im = ax.imshow(corr, cmap="liouai", vmin=0.5, vmax=1)
    ax.set_xticks(range(6), corr.columns, rotation=60, ha="right", fontsize=8)
    ax.set_yticks(range(6), corr.columns, fontsize=8)
    for i in range(6):
        for j in range(6):
            ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if corr.iloc[i, j] > 0.8 else LIOUAI["navy"])
    ax.set_title(nom)
plt.tight_layout(); plt.show()

### Les Médicis

In [ ]:
membres_f, A_f, c_f = resultats["florence"]
print((c_f / c_f.max()).round(2).sort_values("intermédiarité", ascending=False).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
dessiner(A_f, membres_f, c_f["degré"], ax=axes[0], titre="Florence : degré")
dessiner(A_f, membres_f, c_f["intermédiarité"], ax=axes[1], titre="Florence : intermédiarité")
plt.tight_layout(); plt.show()

**Activité** : Retirez les Médicis du réseau (en pensée, puis par le calcul dans l'exercice) : que deviennent les liens entre les autres familles ? Comment l'intermédiarité traduit-elle cette situation ?

Indiquez vos réponses et justifiez-les.

### Les *Misérables*

In [ ]:
membres_m, A_m, c_m = resultats["miserables"]
_, _, A_m_poids = charger("miserables", pondere=True)
propre_pondere = np.abs(np.linalg.eigh(A_m_poids)[1][:, -1])
c_m["vecteur propre (pondéré)"] = propre_pondere / propre_pondere.sum()
print(c_m.sort_values("degré", ascending=False).head(8).round(3).to_string())

groupe, _ = coupe_spectrale(A_m)
print("\nCoupe spectrale :", groupe.sum(), "personnages d'un côté,", len(groupe) - groupe.sum(), "de l'autre")
print("Le plus petit groupe :", ", ".join(membres_m["id"][groupe == (1 if groupe.sum() < len(groupe) / 2 else 0)]))

## ⑤ Interpréter

In [ ]:
print(f"Vos paris : {pari_famille_la_plus_influente}, {pari_personnage_le_plus_influent}")
print("Premiers selon chaque mesure :")
for nom, (_, _, c) in resultats.items():
    print(f"  {nom:10s} :", ", ".join(f"{m} → {c[m].idxmax()}" for m in c.columns[:6]))
moyennes = {nom: c.iloc[:, :6].rank().corr().where(~np.eye(6, dtype=bool)).stack().mean() for nom, (_, _, c) in resultats.items()}
print("\nAccord moyen entre les mesures :", ", ".join(f"{k} {v:.2f}" for k, v in moyennes.items()))

**Activité** :

1. **Conclusions générales.** Rédigez trois ou quatre conclusions qui valent pour les trois réseaux, par exemple sur le lien entre degré et PageRank, ou sur ce que seule l'intermédiarité révèle. Indiquez pour chacune sur quels réseaux elle est vérifiée.
2. **Argumenter.** On affirme : « Les Médicis étaient la famille la plus influente de Florence. » Listez les faits du calcul qui **soutiennent** cette affirmation, puis ceux qui l'**affaiblissent** ou la limitent (nature des liens, mesures où l'écart est faible, ce que les données ne contiennent pas). Concluez en une phrase nuancée.
3. **Pour l'association.** Quelle mesure lui conseillez-vous selon son objectif : diffuser une information, repérer les personnes qui relient des groupes isolés, identifier les « piliers » d'une communauté ?

Indiquez vos réponses et justifiez-les.

### Le rapport et la présentation

Rédigez un rapport de **4 à 6 pages** selon ce plan :

1. **La question** : les sens du mot « influent », les sous-questions, les hypothèses de la représentation en réseau.
2. **Les données** : les trois réseaux, leur origine, leur fiabilité.
3. **Les outils** : chaque mesure en une phrase et un petit schéma ; le vocabulaire (degré, matrice d'adjacence, vecteur propre, laplacien, intermédiarité) défini une seule fois, puis utilisé de façon cohérente.
4. **Les épreuves** : rumeur simulée, scission du club ; ce qu'elles valident et ce qu'elles ne valident pas.
5. **Les conclusions générales**, et l'argumentaire sur les Médicis.
6. **Le mode d'emploi du module** et ses limites.

Pour la présentation de 10 minutes, **6 diapositives au plus**. Choisissez avec soin chaque représentation : un dessin de réseau pour montrer une position, un tableau pour comparer des mesures, une formule seulement si elle est indispensable.

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Le coût.** L'intermédiarité, calculée ici par puissances de matrices et triple boucle, demande de l'ordre de $n^3$ opérations, voire plus. Que se passe-t-il pour un réseau social d'un million de comptes ? Quelles mesures restent calculables ?
- **Les réseaux orientés.** Sur un réseau social où l'on « suit » quelqu'un sans réciprocité, la matrice n'est plus symétrique. Quelles fonctions du module faudrait-il revoir ?
- **Généraliser : le module.** Copiez les fonctions dans un fichier `reseau.py` à côté de `reseaux.py`, et écrivez un fichier `test_reseau.py` qui teste au moins : une étoile, un cycle, un chemin, deux triangles reliés par un lien (coupe spectrale), les bornes de la propagation ($p = 0$ et $p = 1$), et des entrées invalides. Utilisez-le pour mesurer l'effet du retrait des Médicis sur le réseau florentin. Le corrigé se trouve dans le dossier `corrige`.

In [ ]:
# À compléter : créez reseau.py et test_reseau.py, puis lancez vos tests et l'expérience du retrait des Médicis

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
import importlib, sys
sys.path.insert(0, "corrige")
tests = importlib.import_module("test_reseau")
for nom in sorted(dir(tests)):
    if nom.startswith("test_"):
        getattr(tests, nom)()
        print("ok", nom)

garde = (membres_f["id"] != "Medici").to_numpy()
A_sans = A_f[np.ix_(garde, garde)]
D, _ = distances_et_chemins(A_sans)
morceaux = len({tuple(np.isfinite(ligne)) for ligne in D})
print(f"\nSans les Médicis : {int(A_sans.sum() / 2)} liens au lieu de {int(A_f.sum() / 2)}, "
      f"et le réseau se coupe en {morceaux} morceaux")

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Tirer des conclusions générales sur un type de problème": 1,
    "Implémenter un modèle général sous forme de programme robuste": 1,
    "Distinguer les faits qui soutiennent ou affaiblissent un argument": 1,
    "Structurer un rapport ou une présentation": 1,
    "Utiliser vocabulaire, symboles, schémas et code à bon escient": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "B4-generaliser-et-convaincre", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")